# Hiệu chỉnh dữ liệu PM và thời tiết

Notebook tạo một bản dữ liệu mới, không ghi đè dữ liệu raw. `PM1.0` trong yêu cầu được hiểu là cột `PM10` vì dữ liệu nguồn không có cảm biến PM1.0.

- PM2.5 dùng ngưỡng AQI cập nhật năm 2024: https://aqs.epa.gov/aqsweb/documents/codetables/aqi_breakpoints.html
- IQAir Hà Nội dùng để kiểm tra thang AQI+ hiện hành: https://www.iqair.com/air-quality/vietnam/ha-noi/hanoi
- Các mốc thời tiết địa phương lấy từ `data/raw/Book1.xlsx`; các đỉnh của quận khác không được gán thẳng cho Hoàn Kiếm.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from calibrate_iqair_data import calibrate_file, validate_result

source_path = project_root / 'data' / 'raw' / 'data.csv'
weather_baseline_path = project_root / 'data' / 'raw' / 'data_1.csv'
output_path = project_root / 'data' / 'processed' / 'data_iqair_calibrated.csv'
audit_path = project_root / 'data' / 'processed' / 'data_iqair_calibration_summary.csv'


In [ ]:
calibrated, audit = calibrate_file(
    source_path=source_path,
    weather_baseline_path=weather_baseline_path,
    output_path=output_path,
    audit_path=audit_path,
)
audit


In [ ]:
validate_result(calibrated)

print(f'Số dòng: {len(calibrated):,}')
print(f'Khoảng thời gian: {calibrated["Local Time"].iloc[0]} → {calibrated["Local Time"].iloc[-1]}')
display(calibrated[['AQI', 'PM25', 'PM10', 'Precipitation', 'Wind Speed']].describe())


## Lưu ý phương pháp

AQI trong tệp là chỉ số tổng. Vì vậy, nồng độ PM chỉ bị hạ khi chính PM đó sẽ tạo ra một chỉ số con cao hơn AQI tổng. Cách này giữ nguyên các điểm hợp lý và chỉ sửa các điểm bất nhất. PM10 được ràng buộc không thấp hơn PM2.5. Gió giật của bão Yagi không được ghi vào cột tốc độ gió duy trì.
